# Latent space exploration

In [ ]:
import gseapy as gp
import pandas as pd
import numpy as np
import anndata as ad
import scanpy as sc
import anndata as ad
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import colorsys
from adjustText import adjust_text
from itertools import product
import os

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.8,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "xtick.labelsize":    7,
    "ytick.labelsize":    7,
    "axes.titlesize":     8,
    "axes.titleweight":   "bold",
    "axes.labelsize":     7,
    "legend.fontsize":    7,
    "legend.title_fontsize": 8,
    "figure.titlesize":   13,
    "figure.titleweight": "bold",
    "figure.dpi":         300,
})

In [ ]:
adata = ad.read_h5ad("adata.h5ad")

In [ ]:
adata.obs['histology'].value_counts()

In [ ]:
adata.obs['class'] = adata.obs['class'].astype(str)
adata.obs['class'] = adata.obs['class'].replace({'OvaryR': 'Adnexa', 'OvaryL': 'Adnexa', 'Ovary': 'Adnexa'})
adata.obs['PFI'] = adata.obs.PFI.astype(str)
adata.obs['PFI_short_long'] = adata.obs['PFI'].replace({'short': 'short', 'medium': 'long', 'long': 'long'})
adata.obs['histology'] = adata.obs.histology.astype(str)
adata.obs['histology'] = adata.obs['histology'].replace({'Tumor Epithelium': 'Tumor epithelium', 'Mixed TumEpi+Other': 'Mixed', 'Other': 'Stroma'})
adata.obs['patient'] = adata.obs['patient'].astype(str)
adata.obs['anno'] = adata.obs['class'] + '_' + adata.obs['patient'] + '_PFI-' + adata.obs['PFI']

In [ ]:
adata = adata[~adata.obs['class'].isin(['Marker'])].copy()

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="seurat_v3",
    subset=False,
    layer='counts',
    batch_key='patient'
)

In [ ]:
adata_log = adata.copy()
adata_log.X = adata.layers['log-transformed'].copy()

In [ ]:
sc.pp.pca(adata_log)
sc.pp.neighbors(adata_log)
sc.tl.leiden(adata_log)
sc.tl.umap(adata_log, min_dist=0.3, spread=1)

In [ ]:
mpl.rcParams['axes.facecolor']   = 'white'
mpl.rcParams['figure.facecolor'] = 'white'

def add_umap_style(ax, title):
    """Minimal UMAP axis: no ticks, labelled arrows for axes."""
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title(title, pad=6)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    # Small UMAP-style axis arrows
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    xr = (xlim[1] - xlim[0]) * 0.12
    yr = (ylim[1] - ylim[0]) * 0.12
    arrow_kw = dict(arrowstyle='->', color='#444444',
                    lw=0.8, mutation_scale=8)
    ax.annotate('', xy=(xlim[0] + xr, ylim[0]),
                xytext=(xlim[0], ylim[0]),
                arrowprops=arrow_kw, annotation_clip=False)
    ax.annotate('', xy=(xlim[0], ylim[0] + yr),
                xytext=(xlim[0], ylim[0]),
                arrowprops=arrow_kw, annotation_clip=False)
    ax.text(xlim[0] + xr / 2, ylim[0] - (ylim[1]-ylim[0])*0.04,
            'UMAP1', fontsize=6, ha='center', va='top', color='#444444')
    ax.text(xlim[0] - (xlim[1]-xlim[0])*0.03, ylim[0] + yr / 2,
            'UMAP2', fontsize=6, ha='right', va='center',
            color='#444444', rotation=90)

def encode_categorical(labels):
    """Map string labels → integer codes + ordered unique list."""
    uniq = list(dict.fromkeys(labels))       # preserve order, deduplicate
    code = np.array([uniq.index(l) for l in labels])
    return code, uniq

In [ ]:
umap_xy = adata_log.obsm['X_umap']

sample_labels = adata.obs['sample'].astype(str).values
sample_codes, sample_uniq = encode_categorical(sample_labels)
n_samples = len(sample_uniq)
sample_palette = plt.get_cmap('okabe_ito')(np.linspace(0.25, 0.95, n_samples))

fig, ax = plt.subplots(figsize=(2.5, 2))
for i, sample in enumerate(sample_uniq):
    mask = sample_codes == i
    ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
               c=[sample_palette[i]], s=3, linewidths=0,
               alpha=0.8, rasterized=True, label=sample)
add_umap_style(ax, 'Sample')
ax.legend(
    title='TMA', title_fontsize=7, fontsize=6,
    markerscale=2, frameon=False,
    bbox_to_anchor=(1.02, .5), loc='center left', borderaxespad=0.,
)
fig.savefig(f'figures/umap_sample.png',
            bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
umap_xy = adata_log.obsm['X_umap']

labels = adata.obs['patient'].astype(str).values
codes, uniq = encode_categorical(labels)
n = len(uniq)
palette = plt.get_cmap('tab20')(np.linspace(0.2, 0.95, n))

fig, ax = plt.subplots(figsize=(2.5, 2))
for i, label in enumerate(uniq):
    mask = codes == i
    ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
               c=[palette[i]], s=3, linewidths=0,
               alpha=0.8, rasterized=True, label=label)
    # cx, cy = umap_xy[mask, 0].mean(), umap_xy[mask, 1].mean()
    # ax.text(cx, cy, label, fontsize=6, ha='center', va='center',
    #         fontweight='bold', color='white',
    #         bbox=dict(boxstyle='round,pad=0.15', fc=palette[i],
    #                   ec='none', alpha=0.75))
add_umap_style(ax, 'Patient')
ax.legend(
    title='Patient', title_fontsize=7, fontsize=6,
    markerscale=2, frameon=False,
    bbox_to_anchor=(.5, -.12), loc='upper center', borderaxespad=0., ncol=3
)
fig.savefig(f'figures/umap_patient.png',
            bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
umap_xy = adata_log.obsm['X_umap']

labels = adata.obs['class'].astype(str).values
codes, uniq = encode_categorical(labels)
n = len(uniq)
palette = plt.get_cmap('okabe_ito')(np.linspace(0.2, 0.95, n))

fig, ax = plt.subplots(figsize=(2.5, 2))
for i, label in enumerate(uniq):
    mask = codes == i
    ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
               c=[palette[i]], s=3, linewidths=0,
               alpha=0.8, rasterized=True, label=label)
    # cx, cy = umap_xy[mask, 0].mean(), umap_xy[mask, 1].mean()
    # ax.text(cx, cy, label, fontsize=6, ha='center', va='center',
    #         fontweight='bold', color='white',
    #         bbox=dict(boxstyle='round,pad=0.15', fc=palette[i],
    #                   ec='none', alpha=0.75))
add_umap_style(ax, 'Tumor site')
ax.legend(
    title='Site', title_fontsize=7, fontsize=6,
    markerscale=2, frameon=False,
    bbox_to_anchor=(.5, -.12), loc='upper center', borderaxespad=0.,
)
fig.savefig(f'figures/umap_site.png',
            bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
import matplotlib.colors as mcolors

def adjust_lightness(color, factor):
    """factor > 1 lightens, < 1 darkens (blends toward white/black)."""
    r, g, b, a = mcolors.to_rgba(color)
    if factor >= 1:
        t = factor - 1
        return (r + (1 - r) * t, g + (1 - g) * t, b + (1 - b) * t, a)
    else:  # darken: scale toward black
        return (r * factor, g * factor, b * factor, a)

umap_xy = adata_log.obsm['X_umap']

patient = adata.obs['patient'].astype(str).values
site = adata.obs['class'].astype(str).values

p_codes, p_uniq = encode_categorical(patient)
n = len(p_uniq)
palette = plt.get_cmap('tab20')(np.linspace(0.2, 0.95, n))

# adjust these strings to your actual site labels
OMENTAL, ADNEXAL = 'Omentum', 'Adnexa'
LIGHT, DARK = 1.5, 0.6

fig, ax = plt.subplots(figsize=(2.5, 2))
for i, p in enumerate(p_uniq):
    base = palette[i]
    for s, factor in [(OMENTAL, LIGHT), (ADNEXAL, DARK)]:
        mask = (p_codes == i) & (site == s)
        if not mask.any():
            continue
        ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
                   c=[adjust_lightness(base, factor)], s=3, linewidths=0,
                   alpha=0.8, rasterized=True)

add_umap_style(ax, 'Patient / Site')

# patient color + light/dark
from matplotlib.lines import Line2D
patient_handles = [
    Line2D([0], [0], marker='o', color='none', lw=0, markeredgewidth=0,
           markerfacecolor=palette[i], markersize=2, label=p)
    for i, p in enumerate(p_uniq)
]
site_handles = [
    Line2D([0], [0], marker='o', color='none', lw=0, markeredgewidth=0.5,
           markerfacecolor=adjust_lightness((0.4, 0.4, 0.4, 1), LIGHT),
           markersize=2, label='Omental (light)'),
    Line2D([0], [0], marker='o', color='none', lw=0, markeredgewidth=0.5,
           markerfacecolor=adjust_lightness((0.4, 0.4, 0.4, 1), DARK),
           markersize=2, label='Adnexal (dark)'),
]
ax.legend(
    handles=patient_handles + site_handles,
    title='Patient / Site', title_fontsize=7, fontsize=6,
    markerscale=2, frameon=False,
    bbox_to_anchor=(.5, -.12), loc='upper center', borderaxespad=0., ncol=4
)
fig.savefig('figures/umap_patient_site.png', bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
umap_xy = adata_log.obsm['X_umap']

labels = adata.obs['histology'].astype(str).values
codes, uniq = encode_categorical(labels)
n = len(uniq)
palette = plt.get_cmap('okabe_ito')(np.linspace(.15, 0.75, n))

fig, ax = plt.subplots(figsize=(2.5, 2))
for i, label in enumerate(uniq):
    mask = codes == i
    ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
               c=[palette[i]], s=3, linewidths=0,
               alpha=0.8, rasterized=True, label=label)
    # cx, cy = umap_xy[mask, 0].mean(), umap_xy[mask, 1].mean()
    # ax.text(cx, cy, label, fontsize=6, ha='center', va='center',
    #         fontweight='bold', color='white',
    #         bbox=dict(boxstyle='round,pad=0.15', fc=palette[i],
    #                   ec='none', alpha=0.75))
add_umap_style(ax, 'Tissue compartment')
ax.legend(
    title='Compartment', title_fontsize=7, fontsize=6,
    markerscale=2, frameon=False,
    bbox_to_anchor=(.5, -.12), loc='upper center', borderaxespad=0.,
)
fig.savefig(f'figures/umap_histo.png',
            bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
umap_xy = adata_log.obsm['X_umap']

labels = adata.obs['PFI'].astype(str).str.title().values
codes, uniq = encode_categorical(labels)
n = len(uniq)
palette = plt.get_cmap('okabe_ito')(np.linspace(.2, 0.75, n))                
palette = {'Short': '#f58c33', 'Medium': '#2a56b8', 'Long': '#289e99'}

fig, ax = plt.subplots(figsize=(2.5, 2))
for i, label in enumerate(uniq):
    mask = codes == i
    ax.scatter(umap_xy[mask, 0], umap_xy[mask, 1],
               c=palette[label], s=3, linewidths=0,
               alpha=0.8, rasterized=True, label=label)
    # cx, cy = umap_xy[mask, 0].mean(), umap_xy[mask, 1].mean()
    # ax.text(cx, cy, label, fontsize=6, ha='center', va='center',
    #         fontweight='bold', color='white',
    #         bbox=dict(boxstyle='round,pad=0.15', fc=palette[i],
    #                   ec='none', alpha=0.75))
add_umap_style(ax, 'Platinum-free interval')
ax.legend(
    title='PFI', title_fontsize=7, fontsize=6,
    markerscale=2, frameon=False,
    bbox_to_anchor=(.5, -.12), loc='upper center', borderaxespad=0.,
)
fig.savefig(f'figures/umap_pfi.png',
            bbox_inches='tight', dpi=600)
plt.show()